[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/13_Monte_Carlo_Simulation.ipynb)

# Monte Carlo Simulation

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 10

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

Last week you used random numbers to compute integrals and to model random walks. This week you use them to **simulate physical processes** that are random by nature: radioactive decay, particle decays, and (as a teaser) a magnet made of tiny spins. The recipe is always the same: describe each random event by a probability distribution, draw random numbers from that distribution, and let the computer play out many events. Then look at the results as if they were experimental data.

## Learning goals

By the end of this notebook you should be able to:

- explain what a probability density function (PDF) and a cumulative distribution function (CDF) are;
- generate random numbers from any 1D distribution with the **inverse-transform** method and the **accept–reject** method;
- check that your samples follow the intended distribution with a histogram;
- simulate radioactive decay and a decay chain atom by atom, and see the statistical fluctuations;
- verify that waiting times between decays are exponential and that counts per interval follow a Poisson distribution;
- simulate the relativistic decay $\pi^+ \to \mu^+ \nu_\mu$, boost it to the lab frame, and do a simple detector acceptance study.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2027)   # fixed seed: same "random" numbers every time you run

## 1. Probability distributions: a quick recap

A continuous random variable $x$ (for example the time at which an atom decays) is described by a **probability density function** (PDF) $p(x)$:

$$P(a \le x \le b) = \int_a^b p(x)\,dx, \qquad p(x) \ge 0, \qquad \int_{-\infty}^{\infty} p(x)\,dx = 1 .$$

The PDF itself is not a probability; it is a probability *per unit $x$*. Areas under it are probabilities.

The **cumulative distribution function** (CDF) is the probability of getting a value at most $x$:

$$F(x) = P(\text{value} \le x) = \int_{-\infty}^{x} p(x')\,dx' .$$

$F$ climbs from 0 to 1 and never decreases. You already know one PDF from quantum mechanics: $|\psi(x)|^2$ is the probability density for finding a particle at position $x$.

`rng.random(n)` gives $n$ numbers drawn from the **uniform** distribution on $[0, 1)$: $p(u) = 1$ there, so $F(u) = u$. NumPy also has built-in samplers such as `rng.normal` and `rng.exponential`, but physics often needs distributions NumPy does not provide. The next two sections show how to build any distribution out of uniform random numbers.

## 2. The inverse-transform method

**Idea.** Draw $u$ uniformly in $[0,1)$ and solve $F(x) = u$ for $x$:

$$x = F^{-1}(u).$$

Why does this work? The fraction of $u$ values that land below $F(x)$ is exactly $F(x)$, because $u$ is uniform. So the fraction of $x$ values below $x$ is $F(x)$, which is the definition of the CDF we wanted. Picture it this way: where the PDF is large, $F$ is steep, so a wide range of $u$ maps onto a narrow range of $x$, and many samples pile up there.

### Example: the exponential distribution

An unstable particle with mean lifetime $\tau$ decays at a random time $t \ge 0$ with PDF

$$p(t) = \frac{1}{\tau} e^{-t/\tau}.$$

Its CDF is

$$F(t) = \int_0^t \frac{1}{\tau} e^{-t'/\tau}\,dt' = 1 - e^{-t/\tau}.$$

Setting $F(t) = u$ and solving: $e^{-t/\tau} = 1-u$, so

$$t = -\tau \ln(1 - u).$$

Since $1-u$ is also uniform on $(0, 1]$, you will often see $t = -\tau\ln u$ instead. Using $1-u$ avoids $\ln 0$, because `rng.random` can return exactly 0 but never exactly 1.

In [ ]:
tau = 2.197          # muon mean lifetime in microseconds
n = 100_000

u = rng.random(n)
t = -tau * np.log(1 - u)     # inverse transform

print(f"sample mean = {t.mean():.4f} us   (expected tau = {tau} us)")
print(f"sample std  = {t.std():.4f} us   (expected tau = {tau} us for an exponential)")

To check the *shape* and not just the mean, compare a histogram with the PDF. With `density=True`, `ax.hist` divides each bin count by $n \times (\text{bin width})$, so the bar heights estimate $p(t)$ and can be drawn on the same axes as the PDF.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(t, bins=80, range=(0, 15), density=True, alpha=0.6, label="inverse-transform samples")

tt = np.linspace(0, 15, 300)
ax.plot(tt, np.exp(-tt / tau) / tau, "k-", label=r"PDF $e^{-t/\tau}/\tau$")
ax.set_xlabel(r"decay time $t$ ($\mu$s)")
ax.set_ylabel(r"probability density (1/$\mu$s)")
ax.legend()
plt.show()

NumPy's own `rng.exponential(tau, n)` uses essentially this method (with a faster algorithm). It is good practice to check your own sampler against a trusted one:

In [ ]:
t_numpy = rng.exponential(tau, n)
print(f"mine:  mean = {t.mean():.4f}, median = {np.median(t):.4f}")
print(f"NumPy: mean = {t_numpy.mean():.4f}, median = {np.median(t_numpy):.4f}")
print(f"exact median = tau ln 2 = {tau * np.log(2):.4f}")

### Example: a power law

Many spectra in nature follow a **power law**. For instance, the energy spectrum of cosmic rays is roughly $p(E) \propto E^{-\alpha}$ with $\alpha \approx 2.7$ over a huge energy range. Take

$$p(E) = C E^{-\alpha}, \qquad E_\text{min} \le E \le E_\text{max}, \qquad \alpha \ne 1 .$$

Normalizing and integrating gives the CDF

$$F(E) = \frac{E^{1-\alpha} - E_\text{min}^{1-\alpha}}{E_\text{max}^{1-\alpha} - E_\text{min}^{1-\alpha}},$$

and solving $F(E) = u$:

$$E = \left[E_\text{min}^{1-\alpha} + u\left(E_\text{max}^{1-\alpha} - E_\text{min}^{1-\alpha}\right)\right]^{1/(1-\alpha)} .$$

In [ ]:
def sample_power_law(alpha, emin, emax, n, rng):
    """Sample n values from p(E) ~ E**(-alpha) on [emin, emax] (alpha != 1)."""
    u = rng.random(n)
    a = 1 - alpha
    return (emin**a + u * (emax**a - emin**a)) ** (1 / a)

alpha, emin, emax = 2.7, 1.0, 1000.0      # energies in GeV
E = sample_power_law(alpha, emin, emax, 200_000, rng)
print(f"min = {E.min():.3f} GeV, max = {E.max():.1f} GeV, median = {np.median(E):.3f} GeV")

A power law spans many decades, so plot it on log–log axes. Equal-width bins would put almost everything in the first bin; instead use **logarithmic bins** made with `np.logspace(start_exponent, stop_exponent, number)`. `density=True` still works with unequal bin widths, because it divides each bin by its own width.

In [ ]:
bins = np.logspace(0, 3, 31)                 # 10^0 ... 10^3, 30 bins
C = (1 - alpha) / (emax**(1 - alpha) - emin**(1 - alpha))   # normalization constant

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(E, bins=bins, density=True, histtype="step", lw=2, label="samples")
EE = np.logspace(0, 3, 200)
ax.plot(EE, C * EE**(-alpha), "k--", label=r"$C E^{-2.7}$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("energy $E$ (GeV)")
ax.set_ylabel("probability density (1/GeV)")
ax.legend()
plt.show()

On log–log axes a power law is a straight line with slope $-\alpha$. The lowest-probability bins at high energy contain few samples, so they fluctuate more. You will meet that again when fitting histograms.

**When can you use inverse transform?** Whenever you can integrate the PDF to get $F$ *and* invert $F$ in closed form (or cheaply with a root finder). That is often not possible, which is why you need the next method.

### Your turn 1

The polar angle $\theta$ of a direction chosen uniformly over a sphere has PDF $p(\theta) = \tfrac{1}{2}\sin\theta$ on $[0, \pi]$.

1. Derive the CDF $F(\theta)$ and invert it by hand.
2. Write code that draws 100,000 angles with the inverse-transform method.
3. Plot a histogram (`density=True`) together with $\tfrac12 \sin\theta$. What fraction of your angles lie within $30^\circ$ of the $+z$ axis? Compare with the exact answer $F(\pi/6)$.

You will need this result in Section 6.

In [ ]:
# Your code here

## 3. The accept–reject method

**Idea.** Throw points uniformly into a box that encloses the curve $p(x)$, and keep only the points that land under the curve. The $x$ values of the kept points are distributed according to $p(x)$.

Algorithm, for $p(x)$ on $[a, b]$ with $p(x) \le p_\text{max}$ everywhere:

1. Draw $x$ uniform on $[a, b]$ and $y$ uniform on $[0, p_\text{max}]$.
2. If $y < p(x)$, **accept** $x$; otherwise **reject** it.
3. Repeat until you have enough samples.

This is the same as hit-or-miss Monte Carlo integration from last week, except now you keep the "hits" as samples. The fraction of points accepted (the **efficiency**) is the area under the curve divided by the box area: $1 / [p_\text{max}(b - a)]$ for a normalized PDF.

Advantages: you only need to be able to *evaluate* $p(x)$; it does not even need to be normalized. Drawback: if the PDF has a tall narrow peak, most points are rejected and the method is slow.

Below is a vectorized version. It generates a batch of candidates at once, keeps the accepted ones, and repeats until it has enough. `np.concatenate` joins a list of arrays end to end.

In [ ]:
def accept_reject(pdf, a, b, pmax, n, rng):
    """Draw n samples from pdf on [a, b], assuming pdf(x) <= pmax there.

    Returns the samples and the fraction of candidates that were accepted.
    """
    accepted = []
    n_have, n_tried = 0, 0
    while n_have < n:
        x = rng.uniform(a, b, n)            # candidate x values
        y = rng.uniform(0, pmax, n)         # heights in the box
        keep = x[y < pdf(x)]                # boolean mask: points under the curve
        accepted.append(keep)
        n_have += keep.size
        n_tried += n
    samples = np.concatenate(accepted)[:n]  # join batches, trim to exactly n
    return samples, n_have / n_tried

### Example: a particle in a box

*Physics recap.* For a particle confined to $0 \le x \le L$ by infinitely high walls (the "infinite square well"), the stationary states are

$$\psi_n(x) = \sqrt{\frac{2}{L}} \sin\!\left(\frac{n\pi x}{L}\right), \qquad n = 1, 2, 3, \dots$$

and $|\psi_n(x)|^2$ is the probability density for finding the particle at $x$. If you measure the position of many particles, each prepared in the same state, the results are random and follow $|\psi_n|^2$. Let's simulate such measurements for $n = 2$, which has a node in the middle of the box. Here $p_\text{max} = 2/L$.

In [ ]:
L = 1.0     # box width in nm

def psi2_squared(x):
    """|psi_2(x)|^2 for the infinite square well of width L."""
    return (2 / L) * np.sin(2 * np.pi * x / L)**2

x_meas, eff = accept_reject(psi2_squared, 0, L, 2 / L, 50_000, rng)
print(f"accepted fraction = {eff:.4f}   (expected 1/(pmax*L) = {1 / ((2 / L) * L):.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(x_meas, bins=60, density=True, alpha=0.6, label="simulated position measurements")
xx = np.linspace(0, L, 300)
ax.plot(xx, psi2_squared(xx), "k-", label=r"$|\psi_2(x)|^2$")
ax.set_xlabel("position $x$ (nm)")
ax.set_ylabel("probability density (1/nm)")
ax.legend()
plt.show()

print(f"<x>   = {x_meas.mean():.4f} nm  (exact L/2 = {L/2})")
print(f"P(x in middle 10% of box) = {np.mean(np.abs(x_meas - L/2) < 0.05*L):.4f}")

Notice the trick in the last line: `np.mean` of a boolean array gives the fraction of `True` values, because `True` counts as 1 and `False` as 0. Very few particles are found near the node at $x = L/2$.

### Example: photon energies from a blackbody

*Physics recap.* A hot object at temperature $T$ emits photons with a spectrum given by Planck's law. Written in terms of the dimensionless photon energy $x = h\nu / k_B T$, the number of photons per unit $x$ is proportional to

$$p(x) \propto \frac{x^2}{e^x - 1},$$

while the *energy* emitted per unit $x$ is proportional to $x^3/(e^x - 1)$. Let's sample photon energies, i.e. use $p(x) \propto x^2/(e^x-1)$. It is not easy to invert its CDF, but accept–reject only needs the function values. The function is not normalized; that is fine.

The function is negligible beyond $x \approx 25$, so cut there. For $p_\text{max}$, evaluate the function on a fine grid and add a 5% safety margin.

In [ ]:
def planck_photon(x):
    """Unnormalized photon-number spectrum x^2/(e^x - 1)."""
    return x**2 / np.expm1(x)     # expm1(x) = e^x - 1, accurate for small x

xgrid = np.linspace(1e-6, 25, 10_000)
pmax = 1.05 * planck_photon(xgrid).max()
print(f"peak of x^2/(e^x-1) at x = {xgrid[np.argmax(planck_photon(xgrid))]:.3f}, pmax = {pmax:.3f}")

x_phot, eff = accept_reject(planck_photon, 0, 25, pmax, 200_000, rng)
print(f"efficiency = {eff:.3f}")

Theory says the mean photon energy is $\langle h\nu\rangle = \dfrac{\pi^4}{30\,\zeta(3)}\,k_B T \approx 2.701\,k_B T$, where $\zeta(3) \approx 1.2021$. Let's check, and then convert to real units for the surface of the Sun ($T \approx 5772$ K) using `scipy.constants`.

In [ ]:
from scipy import constants as const

print(f"<x> from samples = {x_phot.mean():.4f}   theory = {np.pi**4 / (30 * 1.2020569):.4f}")

T_sun = 5772.0                                       # K
E_phot_eV = x_phot * const.k * T_sun / const.e       # photon energies in eV
print(f"mean solar photon energy ~ {E_phot_eV.mean():.3f} eV")

In [ ]:
from scipy.integrate import trapezoid

# normalize the curve numerically so it can be drawn on top of a density histogram
norm = trapezoid(planck_photon(xgrid), xgrid)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(x_phot, bins=100, range=(0, 15), density=True, alpha=0.6, label="sampled photons")
ax.plot(xgrid, planck_photon(xgrid) / norm, "k-", label=r"$x^2/(e^x-1)$, normalized")
ax.set_xlim(0, 15)
ax.set_xlabel(r"$x = h\nu / k_B T$")
ax.set_ylabel("probability density")
ax.legend()
plt.show()

**Pitfall: $p_\text{max}$ too small.** If your box is lower than the true maximum of $p(x)$, the top of the peak gets cut off and the samples are wrong, with no error message. Always choose $p_\text{max}$ generously (or check it with a fine grid as above). A box that is too tall only costs speed.

### Your turn 2

*Physics recap:* in the ground state of hydrogen, the probability density for finding the electron at distance $r$ from the proton is $P(r) = 4 r^2 e^{-2r}$, with $r$ in units of the Bohr radius $a_0$.

1. Use `accept_reject` to draw 100,000 values of $r$ on $[0, 12]$. Choose $p_\text{max}$ by evaluating $P$ on a grid.
2. Compare a histogram with $P(r)$.
3. Check that the sample mean agrees with $\langle r\rangle = 1.5\,a_0$, and estimate the probability that the electron is farther than $3 a_0$ from the proton.
4. What efficiency did you get? Can you explain the number?

In [ ]:
# Your code here

## 4. Radioactive decay as a random process

*Physics recap.* An unstable nucleus has no memory: in any short time interval $\Delta t$ it decays with the same probability, no matter how long it has already existed. For a mean lifetime $\tau$, the probability to decay during a step $\Delta t$ is

$$p = 1 - e^{-\Delta t/\tau} \approx \frac{\Delta t}{\tau} \quad (\Delta t \ll \tau).$$

Starting with $N_0$ nuclei, the *average* number left at time $t$ is $N_0 e^{-t/\tau}$. The **half-life** is $t_{1/2} = \tau \ln 2$. Real decays are random, so a real sample fluctuates around this curve.

As an example, use thallium-208, which has $t_{1/2} = 3.053$ min (Newman's *Computational Physics*, Example 10.1). Simulate it atom by atom: keep a boolean array `alive`, and at every time step each living atom decays if a uniform random number is below $p$.

In [ ]:
t_half = 3.053 * 60          # seconds
tau_Tl = t_half / np.log(2)  # mean lifetime in seconds
N0 = 1000
dt = 1.0                     # time step in seconds
p_decay = 1 - np.exp(-dt / tau_Tl)
n_steps = 1000

alive = np.ones(N0, dtype=bool)        # True = not yet decayed
N_alive = [N0]
for step in range(n_steps):
    decays = alive & (rng.random(N0) < p_decay)   # living AND unlucky this step
    alive[decays] = False
    N_alive.append(alive.sum())

times = np.arange(n_steps + 1) * dt
print(f"p per step = {p_decay:.5f}; atoms left after {times[-1]:.0f} s: {N_alive[-1]}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(times / 60, N_alive, label="simulation (one sample)")
ax.plot(times / 60, N0 * np.exp(-times / tau_Tl), "k--", label=r"$N_0 e^{-t/\tau}$")
ax.set_xlabel("time (min)")
ax.set_ylabel("number of Tl-208 atoms")
ax.legend()
plt.show()

### A faster way: binomial draws

You do not need to track every atom. If $N$ atoms are alive and each decays with probability $p$, the number that decay is a **binomial** random number: $n$ independent trials, each succeeding with probability $p$. NumPy samples it directly with `rng.binomial(N, p)`. Each step is now one call, regardless of how many atoms there are.

That makes it cheap to repeat the whole experiment many times and look at the **fluctuations**. The cell below runs 200 independent experiments at once, by giving `rng.binomial` an array of 200 values of $N$.

In [ ]:
n_exp = 200
N = np.full(n_exp, N0)                   # 200 experiments, each starting with N0 atoms
history = np.empty((n_steps + 1, n_exp))
history[0] = N
for step in range(1, n_steps + 1):
    N = N - rng.binomial(N, p_decay)     # each experiment loses a binomial number of atoms
    history[step] = N

print("shape of history:", history.shape, "(time steps, experiments)")

How big should the spread be? At time $t$ each original atom has survived with probability $q = e^{-t/\tau}$, independently of the others. That is again a binomial situation, with $N_0$ trials, so the standard deviation of $N(t)$ is

$$\sigma_N = \sqrt{N_0\, q\,(1 - q)} .$$

The relative fluctuation $\sigma_N / N$ grows as the sample shrinks: small samples are noisier.

In [ ]:
q = np.exp(-times / tau_Tl)
mean_theory = N0 * q
sigma_theory = np.sqrt(N0 * q * (1 - q))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(times / 60, history[:, :20], color="0.7", lw=0.5)     # first 20 experiments
ax.plot(times / 60, mean_theory, "k-", label="mean")
ax.fill_between(times / 60, mean_theory - sigma_theory, mean_theory + sigma_theory,
                color="C1", alpha=0.4, label=r"mean $\pm\,\sigma_N$")
ax.set_xlabel("time (min)")
ax.set_ylabel("atoms remaining")
ax.legend()
plt.show()

i = 300   # t = 300 s
print(f"at t = {times[i]:.0f} s: simulated mean = {history[i].mean():.1f}, std = {history[i].std():.2f}")
print(f"               theory:  mean = {mean_theory[i]:.1f}, std = {sigma_theory[i]:.2f}")

`ax.fill_between(x, lower, upper)` shades the region between two curves, a handy way to show a band.

### A third way: sample the decay times directly

Since each atom's decay time follows $p(t) = e^{-t/\tau}/\tau$, you can skip the time steps entirely: draw one decay time per atom with the inverse-transform method from Section 2. The number alive at time $t$ is then the number of decay times larger than $t$. No time-step error, and it is very fast.

In [ ]:
t_decay = -tau_Tl * np.log(1 - rng.random(N0))            # one decay time per atom
N_direct = np.array([np.sum(t_decay > tk) for tk in times])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(times / 60, N_direct, label="decay times sampled directly")
ax.plot(times / 60, N0 * np.exp(-times / tau_Tl), "k--", label=r"$N_0 e^{-t/\tau}$")
ax.set_xlabel("time (min)")
ax.set_ylabel("atoms remaining")
ax.legend()
plt.show()

## 5. A decay chain

Often the daughter of a decay is itself unstable: $A \to B \to C$, with $C$ stable. The number of $B$ nuclei first grows (fed by $A$) and then falls (as $B$ decays). Let's simulate a chain with $\tau_A = 10$ min and $\tau_B = 3$ min. These are made-up values chosen to make the plot clear; real chains such as $^{212}\text{Pb} \to {}^{212}\text{Bi} \to \dots$ behave the same way.

At each step: some $A$ atoms decay into $B$, and (separately) some $B$ atoms decay into $C$. Updating in the right order matters only at the level of one step, which is small.

In [ ]:
tau_A, tau_B = 10.0, 3.0        # minutes
dt = 0.05                       # minutes
pA = 1 - np.exp(-dt / tau_A)
pB = 1 - np.exp(-dt / tau_B)

NA, NB, NC = 10_000, 0, 0
t_chain = np.arange(0, 60 + dt, dt)
counts = np.empty((t_chain.size, 3))
for k in range(t_chain.size):
    counts[k] = NA, NB, NC
    dA = rng.binomial(NA, pA)   # A -> B this step
    dB = rng.binomial(NB, pB)   # B -> C this step
    NA, NB, NC = NA - dA, NB + dA - dB, NC + dB

print("final A, B, C:", counts[-1], " total =", counts[-1].sum())

The average behavior follows from the rate equations $dN_A/dt = -N_A/\tau_A$ and $dN_B/dt = N_A/\tau_A - N_B/\tau_B$ (an ODE system you could also solve with `solve_ivp`). With $\lambda = 1/\tau$ the solution is (the **Bateman equations**)

$$N_A = N_0 e^{-\lambda_A t}, \qquad N_B = N_0 \frac{\lambda_A}{\lambda_B - \lambda_A}\left(e^{-\lambda_A t} - e^{-\lambda_B t}\right), \qquad N_C = N_0 - N_A - N_B .$$

In [ ]:
lA, lB, N0c = 1 / tau_A, 1 / tau_B, 10_000
NA_th = N0c * np.exp(-lA * t_chain)
NB_th = N0c * lA / (lB - lA) * (np.exp(-lA * t_chain) - np.exp(-lB * t_chain))
NC_th = N0c - NA_th - NB_th

fig, ax = plt.subplots(figsize=(6, 4))
for col, (name, th) in enumerate(zip(["A", "B", "C"], [NA_th, NB_th, NC_th])):
    ax.plot(t_chain, counts[:, col], color=f"C{col}", label=f"{name} (simulation)")
    ax.plot(t_chain, th, "k--", lw=1)
ax.plot([], [], "k--", lw=1, label="Bateman solution")    # empty plot: legend entry only
ax.set_xlabel("time (min)")
ax.set_ylabel("number of nuclei")
ax.legend()
plt.show()

### Your turn 3

Many real nuclei can decay in more than one way (**branching**). Modify the chain so that each $A$ decay produces $B$ with probability 0.64 and a different stable nucleus $D$ with probability 0.36 (these are the branching fractions of $^{212}\text{Bi}$).

Hint: first draw the number of $A$ decays in the step with `rng.binomial`, then split those decays between $B$ and $D$ with a second `rng.binomial`.

Plot $N_A, N_B, N_C, N_D$ versus time and check that $N_D/(N_C + N_D)$ approaches 0.36 at late times.

In [ ]:
# Your code here

## 6. Waiting times and counting statistics

Put a Geiger counter next to a long-lived source, so the decay rate $R$ (decays per second) is essentially constant during the measurement. Two questions:

1. What is the distribution of the **waiting time** between successive clicks?
2. If you count clicks in fixed one-second intervals, what is the distribution of the **counts**?

The answers are the two most important distributions in counting experiments:

- Because decays are memoryless, waiting times are **exponential**: $p(\Delta t) = R e^{-R\Delta t}$.
- The number of counts $k$ in an interval of length $T$ follows the **Poisson distribution** with mean $\mu = RT$:
$$P(k) = \frac{\mu^k e^{-\mu}}{k!}, \qquad k = 0, 1, 2, \dots$$
  A key property: its variance equals its mean, so the standard deviation is $\sqrt{\mu}$. This is where the familiar "$\sqrt{N}$ error" on a count comes from.

Let's verify both by simulation. Generate exponential waiting times, then add them up with `np.cumsum` (cumulative sum: element $k$ is the sum of the first $k+1$ elements) to get the click times.

In [ ]:
R = 4.0                       # mean rate, clicks per second
waits = rng.exponential(1 / R, 20_000)
click_times = np.cumsum(waits)
print("first five click times (s):", np.round(click_times[:5], 3))
print(f"total duration ~ {click_times[-1]:.0f} s")

To count clicks per second, histogram the click times with bins one second wide (`np.histogram` returns the counts and the bin edges without plotting). Then count how often each value $k$ occurs, using `np.bincount`, which counts how many times each non-negative integer appears in an array.

`scipy.stats.poisson.pmf(k, mu)` evaluates the Poisson probability $P(k)$ ("pmf" = probability mass function, the discrete version of a PDF).

In [ ]:
from scipy.stats import poisson

T_total = int(click_times[-1])
counts_per_s, _ = np.histogram(click_times, bins=np.arange(0, T_total + 1, 1.0))
print(f"{counts_per_s.size} one-second intervals; mean = {counts_per_s.mean():.3f}, variance = {counts_per_s.var():.3f}")

k = np.arange(counts_per_s.max() + 1)
observed_freq = np.bincount(counts_per_s) / counts_per_s.size

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(waits, bins=60, range=(0, 2), density=True, alpha=0.6, label="simulated waits")
w = np.linspace(0, 2, 200)
axes[0].plot(w, R * np.exp(-R * w), "k-", label=r"$R e^{-R\,\Delta t}$")
axes[0].set_xlabel(r"waiting time $\Delta t$ (s)")
axes[0].set_ylabel("probability density (1/s)")
axes[0].legend()

axes[1].bar(k, observed_freq, alpha=0.6, label="simulated counts")
axes[1].plot(k, poisson.pmf(k, R), "ko", label=f"Poisson, $\\mu = {R}$")
axes[1].set_xlabel("counts in a 1 s interval, $k$")
axes[1].set_ylabel("probability")
axes[1].legend()
plt.tight_layout()
plt.show()

The mean and the variance of the counts agree, as Poisson statistics demands.

### Your turn 4

Repeat the counting with **0.1-second** intervals (mean $\mu = 0.4$) and with **10-second** intervals (mean $\mu = 40$). For each, compare the observed frequencies with `poisson.pmf`.

For $\mu = 40$, also overlay a Gaussian with mean $\mu$ and standard deviation $\sqrt{\mu}$. For which $\mu$ is the Gaussian a good approximation to the Poisson distribution, and for which is it bad? (Think about what a Gaussian would predict for negative counts.)

In [ ]:
# Your code here

## 7. A relativistic two-body decay: $\pi^+ \to \mu^+ \nu_\mu$

Now a particle-physics simulation. Accelerator neutrino beams are made exactly like this: protons hit a target, produce pions, and the pions decay in flight in a long pipe into muons and neutrinos.

*Physics recap (special relativity).* We use units with $c = 1$: energies, momenta, and masses are all in MeV (strictly MeV, MeV/$c$, MeV/$c^2$). For a particle of mass $m$, energy $E$ and momentum $p$:

$$E^2 = p^2 + m^2, \qquad \gamma = \frac{E}{m}, \qquad \beta = \frac{v}{c} = \frac{p}{E}.$$

Energy and momentum are both conserved in a decay.

Masses and lifetime (Particle Data Group, pdg.lbl.gov): $m_\pi = 139.57$ MeV, $m_\mu = 105.66$ MeV, neutrino mass $\approx 0$, charged pion mean lifetime $\tau_\pi = 26.03$ ns.

### Step 1: the decay in the pion rest frame

In the pion's rest frame the pion has $E = m_\pi$ and $p = 0$, so the muon and neutrino fly off back to back with equal and opposite momenta of size $p^*$ (stars denote rest-frame quantities). Energy conservation reads $m_\pi = \sqrt{p^{*2} + m_\mu^2} + p^*$. Solving for $p^*$:

$$p^* = \frac{m_\pi^2 - m_\mu^2}{2 m_\pi}, \qquad E_\mu^* = \frac{m_\pi^2 + m_\mu^2}{2 m_\pi} .$$

Every decay gives the same $p^*$; only the **direction** is random.

In [ ]:
m_pi = 139.57    # MeV (PDG)
m_mu = 105.66    # MeV (PDG)
tau_pi = 26.03e-9  # s (PDG)

# cross-check the muon mass against scipy.constants (CODATA)
print("CODATA muon mass:", const.physical_constants["muon mass energy equivalent in MeV"][0], "MeV")

p_star = (m_pi**2 - m_mu**2) / (2 * m_pi)
E_star = (m_pi**2 + m_mu**2) / (2 * m_pi)
print(f"p* = {p_star:.3f} MeV,  E*_mu = {E_star:.3f} MeV,  muon kinetic energy = {E_star - m_mu:.3f} MeV")

### Step 2: random directions, done right

The decay has no preferred direction, so the muon direction is **isotropic**: uniform over the sphere. In spherical coordinates the direction is given by the polar angle $\theta$ and the azimuthal angle $\phi$. It is tempting to pick $\theta$ uniform in $[0, \pi]$ and $\phi$ uniform in $[0, 2\pi)$. **That is wrong.**

The reason is the area element on the unit sphere, $dA = \sin\theta\, d\theta\, d\phi$. A band near the equator ($\theta \approx \pi/2$) has much more area than a band of the same $d\theta$ near the poles. Uniform $\theta$ would put equal numbers of points in each band, overcrowding the poles.

Fix: notice $\sin\theta\,d\theta = -d(\cos\theta)$, so $dA = d(\cos\theta)\, d\phi$. The area is uniform in $\cos\theta$. So draw

$$\cos\theta \sim \text{uniform}[-1, 1], \qquad \phi \sim \text{uniform}[0, 2\pi).$$

(This is what you derived in Your turn 1.) Let's compare the two methods by looking at $z = \cos\theta$, which should be flat for an isotropic distribution.

In [ ]:
n_dec = 200_000
theta_wrong = rng.uniform(0, np.pi, n_dec)
cos_right = rng.uniform(-1, 1, n_dec)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].hist(np.cos(theta_wrong), bins=50, density=True)
axes[0].set_title(r"WRONG: $\theta$ uniform")
axes[1].hist(cos_right, bins=50, density=True, color="C2")
axes[1].set_title(r"RIGHT: $\cos\theta$ uniform")
for ax in axes:
    ax.set_xlabel(r"$z = \cos\theta$")
axes[0].set_ylabel("probability density")
plt.show()

The wrong method piles directions up near the poles ($z = \pm 1$).

### Step 3: the muon momentum in the pion rest frame

In [ ]:
cos_ts = rng.uniform(-1, 1, n_dec)          # cos(theta*) in pion rest frame
phi_s = rng.uniform(0, 2 * np.pi, n_dec)
sin_ts = np.sqrt(1 - cos_ts**2)

px_s = p_star * sin_ts * np.cos(phi_s)
py_s = p_star * sin_ts * np.sin(phi_s)
pz_s = p_star * cos_ts

### Step 4: boost to the lab

*Physics recap (Lorentz transformation).* If the pion moves along $+z$ in the lab with speed $\beta$ and Lorentz factor $\gamma$, a muon with rest-frame energy $E^*$ and momentum components $(p_x^*, p_y^*, p_z^*)$ has lab values

$$E = \gamma\,(E^* + \beta p_z^*), \qquad p_z = \gamma\,(p_z^* + \beta E^*), \qquad p_x = p_x^*, \quad p_y = p_y^* .$$

Momentum perpendicular to the boost is unchanged. Take a pion beam with momentum $p_\pi = 10$ GeV = 10,000 MeV along $z$.

In [ ]:
p_pi = 10_000.0                       # MeV
E_pi = np.sqrt(p_pi**2 + m_pi**2)
gamma = E_pi / m_pi
beta = p_pi / E_pi
print(f"pion: E = {E_pi:.1f} MeV, gamma = {gamma:.2f}, beta = {beta:.6f}")

E_mu = gamma * (E_star + beta * pz_s)
pz_mu = gamma * (pz_s + beta * E_star)
px_mu, py_mu = px_s, py_s

**Validate before you trust.** Two checks that must hold for every event: the muon is still on its mass shell ($E^2 - p^2 = m_\mu^2$), and the muon plus neutrino add up to the pion's four-momentum. The neutrino has the opposite rest-frame momentum, so boost $(p^*, -\vec p^*)$ the same way.

In [ ]:
m_mu_check = np.sqrt(E_mu**2 - px_mu**2 - py_mu**2 - pz_mu**2)
print(f"muon mass from lab 4-vectors: min {m_mu_check.min():.4f}, max {m_mu_check.max():.4f} MeV")

E_nu = gamma * (p_star - beta * pz_s)          # neutrino: E* = p*, p_z* = -pz_s
pz_nu = gamma * (-pz_s + beta * p_star)
print("max |E_mu + E_nu - E_pi|   =", np.abs(E_mu + E_nu - E_pi).max(), "MeV")
print("max |pz_mu + pz_nu - p_pi| =", np.abs(pz_mu + pz_nu - p_pi).max(), "MeV")

The differences are at the level of floating-point round-off. Good.

### Step 5: the lab muon energy spectrum

Because $E = \gamma(E^* + \beta p^* \cos\theta^*)$ is a *linear* function of $\cos\theta^*$, and $\cos\theta^*$ is uniform, the lab energy is **uniform** between

$$E_\text{min} = \gamma(E^* - \beta p^*), \qquad E_\text{max} = \gamma(E^* + \beta p^*).$$

In [ ]:
E_min = gamma * (E_star - beta * p_star)
E_max = gamma * (E_star + beta * p_star)
print(f"kinematic limits: {E_min/1000:.3f} GeV to {E_max/1000:.3f} GeV")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(E_mu / 1000, bins=60, density=True, alpha=0.6, label="simulation")
ax.hlines(1000 / (E_max - E_min), E_min / 1000, E_max / 1000, colors="k", label="flat prediction")
ax.set_xlabel(r"lab muon energy $E_\mu$ (GeV)")
ax.set_ylabel("probability density (1/GeV)")
ax.legend(loc="lower center")
plt.show()

The muon takes between 57% and 100% of the pion energy, simply because $m_\mu$ is close to $m_\pi$. The neutrino gets the rest, which is why neutrino beams from pion decay have lower energy than the pions.

### Step 6: lab angles

The lab angle of the muon relative to the beam is $\theta_\text{lab} = \arctan(p_T / p_z)$, with transverse momentum $p_T = \sqrt{p_x^2 + p_y^2}$. Use `np.arctan2(pT, pz)`, which handles all quadrants correctly. Since $p_T \le p^* \approx 30$ MeV while $p_z$ is several GeV, the angles are tiny: milliradians.

In [ ]:
pT_mu = np.sqrt(px_mu**2 + py_mu**2)
theta_lab = np.arctan2(pT_mu, pz_mu)        # radians

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(theta_lab * 1000, bins=80, density=True)
ax.set_xlabel(r"muon lab angle $\theta_\mathrm{lab}$ (mrad)")
ax.set_ylabel("probability density (1/mrad)")
plt.show()
print(f"largest lab angle: {theta_lab.max()*1000:.2f} mrad")

Even muons emitted *backwards* in the pion frame move forward in the lab: the pion is moving faster than the muon moves in the pion frame, so the boost wins. There is a maximum lab angle, and the distribution piles up just below it (a "Jacobian peak").

### Step 7: where does the pion decay? Time dilation

The pion lives a random **proper time** $t^*$ drawn from an exponential with mean $\tau_\pi$. In the lab its clock runs slow by a factor $\gamma$, so it travels a distance

$$\ell = \beta c \,(\gamma t^*), \qquad \langle \ell \rangle = \gamma \beta c \tau_\pi = \frac{p_\pi}{m_\pi} c\tau_\pi .$$

Without time dilation the mean would be only $\beta c \tau_\pi \approx 7.8$ m.

In [ ]:
t_proper = rng.exponential(tau_pi, n_dec)          # s
ell = beta * const.c * gamma * t_proper             # m

mean_len = gamma * beta * const.c * tau_pi
print(f"gamma*beta*c*tau = {mean_len:.1f} m;   sample mean = {ell.mean():.1f} m")
print(f"without time dilation: beta*c*tau = {beta * const.c * tau_pi:.2f} m")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ell, bins=80, range=(0, 3000), density=True, alpha=0.6, label="simulated decay points")
ll = np.linspace(0, 3000, 300)
ax.plot(ll, np.exp(-ll / mean_len) / mean_len, "k-", label=r"$e^{-\ell/\gamma\beta c\tau}/\gamma\beta c\tau$")
ax.set_xlabel(r"decay length $\ell$ (m)")
ax.set_ylabel("probability density (1/m)")
ax.legend()
plt.show()

A 10 GeV pion travels on average more than half a kilometer before decaying. That is why neutrino facilities have decay pipes hundreds of meters long. For example, the fraction decaying in the first 200 m is:

In [ ]:
L_pipe = 200.0   # m
print(f"simulated: {np.mean(ell < L_pipe):.4f},   exact 1 - exp(-L/<l>) = {1 - np.exp(-L_pipe / mean_len):.4f}")

### Step 8: a simple detector acceptance study

A real detector covers only part of the space around the decay. The **acceptance** is the fraction of events the detector can see. Suppose a muon detector downstream accepts muons within $\theta_\text{lab} < 1.0$ mrad of the beam axis. What fraction of muons does it catch, and does acceptance bias the energy spectrum?

In [ ]:
theta_cut = 1.0e-3          # rad
inside = theta_lab < theta_cut
print(f"acceptance = {inside.mean():.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
bins = np.linspace(E_min, E_max, 41) / 1000
ax.hist(E_mu / 1000, bins=bins, histtype="step", lw=2, label="all muons")
ax.hist(E_mu[inside] / 1000, bins=bins, histtype="step", lw=2, label=r"accepted ($\theta < 1$ mrad)")
ax.set_xlabel(r"lab muon energy $E_\mu$ (GeV)")
ax.set_ylabel("muons per bin")
ax.legend(loc="upper left")
plt.show()

The detector preferentially sees the **high-energy** muons, the ones emitted forward in the pion frame, since those have the smallest lab angles. If you measured the energy spectrum with this detector and forgot about acceptance, you would get the wrong answer. Simulations like this are how experiments correct for their detectors.

### Your turn 5

1. Write a function `acceptance(p_pi, theta_cut, n, rng)` that returns the fraction of muons with $\theta_\text{lab} <$ `theta_cut` for a pion beam momentum `p_pi` (in MeV). Reuse the steps above.
2. Plot the acceptance for $\theta_\text{cut} = 1$ mrad as a function of $p_\pi$ from 1 GeV to 50 GeV. Explain the trend in a sentence.
3. Plot a histogram of the lab **neutrino** energy for $p_\pi = 10$ GeV. What are its kinematic limits?

In [ ]:
# Your code here

## 8. Optional teaser: the Metropolis algorithm and the Ising model

So far each random event was independent. Many systems in statistical physics need a different kind of simulation: one that samples **configurations** of many interacting parts with the Boltzmann probability $\propto e^{-E/k_B T}$. The number of configurations is astronomically large, so you cannot list them. The **Metropolis algorithm** instead takes a random walk through configuration space that visits each configuration with the right probability.

*Physics recap.* The 2D **Ising model** is a toy magnet: a square grid of spins $s_{ij} = \pm 1$ (up or down). Neighboring spins prefer to point the same way. With coupling $J$ the energy is

$$E = -J \sum_{\langle ij \rangle} s_i s_j ,$$

summed over nearest-neighbor pairs. At low temperature the spins align (a magnet); at high temperature they are disordered. In 2D the transition happens at $k_B T_c / J = 2/\ln(1+\sqrt2) \approx 2.269$.

**Metropolis step:** start from some configuration (below: all spins up). Pick a random spin; compute the energy change $\Delta E$ if it flipped. If $\Delta E \le 0$, flip it. Otherwise flip it with probability $e^{-\Delta E / k_B T}$. One **sweep** is as many steps as there are spins. Units below: $J = k_B = 1$.

In [ ]:
def ising_metropolis(L, T, n_sweeps, rng):
    """2D Ising model on an L x L grid with periodic boundaries, Metropolis updates.
    Returns the final spin grid and the magnetization per spin after each sweep."""
    s = np.ones((L, L), dtype=int)       # "cold start": all spins up
    mags = []
    for sweep in range(n_sweeps):
        sites = rng.integers(0, L, size=(L * L, 2))   # random sites for this sweep
        r = rng.random(L * L)
        for k in range(L * L):
            i, j = sites[k]
            nb_sum = s[(i+1) % L, j] + s[i-1, j] + s[i, (j+1) % L] + s[i, j-1]   # % L wraps around
            dE = 2 * s[i, j] * nb_sum
            if dE <= 0 or r[k] < np.exp(-dE / T):
                s[i, j] = -s[i, j]
        mags.append(s.mean())
    return s, np.array(mags)

A note on the neighbor sum: `s[i-1, j]` with `i = 0` gives `s[-1, j]`, the last row, which is exactly the periodic wrap-around we want. Going the other way, `(i+1) % L` wraps `L` back to `0` (`%` is the remainder operator).

Pure-Python loops are slow, so keep the grid small. This takes a few seconds:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, T in zip(axes, [1.5, 2.27, 3.5]):
    s, mags = ising_metropolis(32, T, 150, rng)
    ax.imshow(s, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"$T$ = {T},  |m| = {abs(mags[-50:]).mean():.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

All three runs started fully magnetized. Below $T_c$ the magnet stays ordered, with only a few flipped spins, and the magnetization per spin $|m|$ stays close to 1. Near $T_c$ you see domains of all sizes. Above $T_c$ the order melts completely: the spins are noisy and $|m| \approx 0$. (If you instead start from random spins at low $T$, domains form and grow, but on a big grid it can take a very long time for one domain to win.) If this interests you, the Ising model makes an excellent final project: measure $|m|(T)$, the energy and the heat capacity, and see how the transition sharpens as the grid grows.

## Summary

- A **PDF** $p(x)$ gives probability per unit $x$; the **CDF** $F(x)$ is its integral, rising from 0 to 1.
- **Inverse transform:** $x = F^{-1}(u)$ with $u$ uniform. Exponential: $t = -\tau\ln(1-u)$. Use it when $F$ can be inverted.
- **Accept–reject:** throw points in a box, keep the ones under $p(x)$. Works for any PDF you can evaluate; efficiency = area under curve / box area. Never let $p_\text{max}$ be too small.
- Always **validate** samples: histogram with `density=True` vs the PDF, and compare means/medians with exact values.
- **Radioactive decay:** each atom decays with probability $p = 1 - e^{-\Delta t/\tau}$ per step; `rng.binomial(N, p)` does a whole population at once. Fluctuations: $\sigma_N = \sqrt{N_0 q(1-q)}$.
- **Waiting times** between random events are exponential; **counts** per interval are Poisson with variance = mean, hence $\sigma = \sqrt{N}$.
- **Isotropic directions:** $\cos\theta$ uniform in $[-1, 1]$, $\phi$ uniform in $[0, 2\pi)$. Not $\theta$ uniform.
- **Two-body decay:** fixed $p^*$ in the parent frame; boost with $E = \gamma(E^* + \beta p_z^*)$, $p_z = \gamma(p_z^* + \beta E^*)$. Check 4-momentum conservation. Lab energy of $\pi\to\mu\nu$ muons is flat; decay length is exponential with mean $\gamma\beta c\tau$.
- **Acceptance** studies: apply the detector's cuts to simulated events and see what fraction and which events survive.
- New tools: `np.cumsum`, `np.bincount`, `np.histogram`, `np.logspace`, `np.concatenate`, `np.arctan2`, `ax.fill_between`, `rng.binomial`, `rng.choice`, `scipy.stats.poisson.pmf`.

## Further reading

- M. Newman, *Computational Physics*, Chapter 10 (random processes and Monte Carlo methods), including the Tl-208 and Bi-213 decay examples and the Ising model.
- NumPy random generator documentation: https://numpy.org/doc/stable/reference/random/generator.html
- Particle Data Group, *Review of Particle Physics*, section "Kinematics" (two-body decays, Lorentz boosts): https://pdg.lbl.gov